# 食材価格

担当：高居
出力: `data/prices.csv`（列は team_rules.md 第2節）、`data/item_map.csv`（食材名の対応表）、`docs/price_trend.png`（価格推移）

## 第3回でやったこと（口頭説明用）

**目的**　外食と自炊のコストを比べるには、食材の値段が要る。このノートブックは、その価格データを作る。

**データ元**　e-Stat「小売物価統計調査（動向編）」の「主要品目の都市別小売価格」。東京都区部（表の名前は「特別区部」）、月次。

**手順**
1. e-Stat から CSV をダウンロードする（`data/raw/` に置く。コミットしない）。
2. **前処理**（`prices.csv` を作るまで）
   - 横持ちの表（列=銘柄、行=月）を、1行=1品目×1か月の縦持ちに直す
   - 銘柄名から番号と【調査終了】などの注記を取る。カンマ付きの数字を数値にする
   - 直近12か月に絞る。12か月のどこかに欠損がある品目（さんま・かつお・豚肉(もも)）は、補完せず除外する
   - 料理に使う22品目に絞り、販売単位（5kg・100g など）を付ける
3. **食材名の対応表**を作る（レシピの食材名 ↔ e-Stat の銘柄名 ↔ 単位）。`data/item_map.csv`
4. **価格推移**を出す（2025-09 を 100 にした指数）。`docs/price_trend.png`
5. 出典を書く（一番下）。

**結果**　22品目 × 12か月 = 264行。

**単位**　総務省の「品目・銘柄・単位」一覧（2025年12月現在）で、22品目すべて確認した。

**追加分**　食パン・豆腐・トマト・ねぎは最初のCSVに入っていなかった。同じ表（e-Stat の画面）から特別区部の値を読み取り、`data/raw/estat_extra_items.csv` に保存して合わせた（直近12か月は 2025-09〜2026-08 にそろえた）。

**課題**　外食の価格は王さんの担当（同じ表に外食の銘柄もある）。

**push 前に「Clear All Outputs」を実行する**

> **このセル**：ファイルの場所、使う22品目、単位を最初にまとめて決めている。
>
> **聞かれたら**：単位（UNIT_MAP）は「その価格が何g・何個ぶんの値段か」。これがないと1gあたりの値段に直せない。総務省の「品目・銘柄・単位」一覧で全部確認した。

In [ ]:
# ===== 設定セル：ファイルの場所・対象品目・単位などの「決め事」をここにまとめる =====
# （この下の関数セルは、ここで決めた値を引数として受け取って動く）
import re                      # 文字列のパターン処理（銘柄名の掃除・月の抜き出し）に使う
import csv                     # e-Stat の CSV を1行ずつ読むために使う
from pathlib import Path       # ファイルの場所（パス）を安全に扱う

import pandas as pd            # 表（DataFrame）を扱う。価格表の加工に使う

# --- 入力・出力ファイルの場所（ノートブックは src/ から実行する前提。team_rules 第3節・optimize.ipynb と同じ） ---
# RAW_PATH：e-Stat からダウンロードした生データ（特別区部・月次・53品目ぶん）。Shift-JIS の CSV
RAW_PATH = Path("../data/raw/FEH_00200571_261016144022.csv")
# EXTRA_RAW_PATH：上の CSV に入っていなかった4品目（食パン・豆腐・トマト・ねぎ）を、
#                 e-Stat の画面から読み取って保存した追加データ。列は raw_name, month, value
EXTRA_RAW_PATH = Path("../data/raw/estat_extra_items.csv")
# OUT_PATH：完成した価格表（prices.csv）の保存先。チームの共通スキーマ（team_rules 第2節）
OUT_PATH = Path("../data/prices.csv")

N_MONTHS = 12        # 何か月ぶん使うか（直近12か月）
CATEGORY = "食材"    # prices.csv の category 列に入れる値（外食は王さんが "外食" で作る）

# USE_KEYWORDS：レシピで使う品目の名前（e-Stat の銘柄名にこの文字が含まれるものだけ残す）
USE_KEYWORDS = ["うるち米", "スパゲッティ", "ゆでうどん", "鶏肉", "豚肉", "鶏卵", "納豆", "さけ",
                "キャベツ", "たまねぎ", "じゃがいも", "にんじん", "もやし", "みそ", "しょう油", "食用油",
                "牛乳", "調理パン", "食パン", "豆腐", "トマト", "ねぎ"]

# UNIT_MAP：{e-Stat の品目番号: 価格がカバーする量}。
#   例：1001（うるち米）の価格は「5kg ぶんの値段」。この単位が無いと円/g に直せない。
UNIT_MAP = {
    # 先に調べた分
    "1001": "5kg", "1042": "1kg", "1031": "1kg", "1221": "100g",
    "1211": "100g", "1341": "10個", "1303": "1000mL", "1106": "100g",
    "1401": "1kg", "1412": "1kg", "1415": "1kg",
    # 総務省「品目・銘柄・単位」一覧（2025年12月現在）で確認した分
    "1407": "1kg", "1417": "1kg", "1473": "150g", "1792": "100g",
    "1601": "900g", "1621": "450mL", "1631": "750g",
    "1021": "1kg", "1471": "1kg", "1436": "1kg", "1405": "1kg",
}

> **このセル**：e-Statの長い銘柄名と、レシピで使う短い呼び名の対応。グラフで色を付ける4品目もここで決めている。
>
> **聞かれたら**：例：「うるち米(単一原料米,「コシヒカリ」)」→「うるち米」。

In [ ]:
# ===== 設定セル2：対応表とグラフ用の決め事 =====

# SHORT_NAMES：{e-Stat の品目番号: レシピで使う短い食材名}
#   e-Stat の銘柄名は長い（例「うるち米(単一原料米,「コシヒカリ」)」）ので、レシピ側の呼び名に対応付ける。
#   recipes.ipynb の INGREDIENT_TO_ITEM と同じ対応（向きが逆）。
SHORT_NAMES = {
    "1001": "うるち米", "1031": "ゆでうどん", "1042": "スパゲッティ", "1106": "さけ",
    "1211": "豚肉", "1221": "鶏肉", "1303": "牛乳", "1341": "鶏卵",
    "1401": "キャベツ", "1407": "もやし", "1412": "じゃがいも", "1415": "にんじん",
    "1417": "たまねぎ", "1473": "納豆", "1601": "食用油", "1621": "しょう油",
    "1631": "みそ", "1792": "調理パン",
    "1021": "食パン", "1471": "豆腐", "1436": "トマト", "1405": "ねぎ",
}

# UNIT_NOTES：{品目番号: (単位の確認状況, 備考)}。item_map.csv の unit_status / note 列になる。
#   出典: 総務省統計局「小売物価統計調査（動向編）品目・銘柄・単位」2025年12月現在。
#   「確認済み」＝公式の一覧で単位を確認した。確認できていない品目は「未確認」と書く。
UNIT_NOTES = {
    "1001": ("確認済み", "1袋 5kg。国産・コシヒカリ"), "1031": ("確認済み", "1kg換算。1食入り(180〜220g)"),
    "1042": ("確認済み", "1kg換算。袋入り(600〜700g)"), "1106": ("確認済み", "100g。トラウトサーモン等・刺身用"),
    "1211": ("確認済み", "100g。国産・バラ"), "1221": ("確認済み", "100g。ブロイラー・もも肉"),
    "1303": ("確認済み", "1本 1000mL。紙容器"), "1341": ("確認済み", "1パック 10個。白色卵"),
    "1401": ("確認済み", "1kg"), "1407": ("確認済み", "1kg。緑豆もやし"),
    "1412": ("確認済み", "1kg"), "1415": ("確認済み", "1kg"), "1417": ("確認済み", "1kg。赤たまねぎを除く"),
    "1473": ("確認済み", "1パック。50g×3個=150g"), "1601": ("確認済み", "1本 900g。キャノーラ油"),
    "1621": ("確認済み", "1本 450mL。こいくち・生しょうゆ"), "1631": ("確認済み", "1個 750g。米みそ"),
    "1792": ("確認済み", "100g。サンドイッチ（タマゴサンド等）"),
    "1021": ("確認済み", "1kg。普通品"), "1471": ("確認済み", "1kg。木綿豆腐・並"),
    "1436": ("確認済み", "1kg。ミニトマトを除く"), "1405": ("確認済み", "1kg。白ねぎ"),
}

# 出力ファイルの場所
OUT_MAP_PATH = Path("../data/item_map.csv")      # 食材の対応表（レシピ名 ↔ e-Stat銘柄 ↔ 単位）
OUT_FIG_PATH = Path("../docs/price_trend.png")   # 価格推移グラフの画像

# HIGHLIGHT：価格推移グラフで色を付ける4品目（{食材名: 色コード}）。他の品目は灰色の線にする。
HIGHLIGHT = {"キャベツ": "#2a78d6", "うるち米": "#eb6834", "じゃがいも": "#1baf7a", "にんじん": "#eda100"}

> **このセル**：e-StatのCSVを読んで、価格表に整える関数たち（ここは定義だけ）。
>
> **聞かれたら**：CSVは横に品目が並ぶクロス集計で、文字コードはShift-JIS。これを「1行＝1品目×1か月」の縦長に組み替える。build_prices は、直近12か月に絞る → 1か月でも欠けた品目は外す → 22品目に絞る → 単位を付ける、の順。欠損を平均で埋めると実際の値動きとずれるので、補完せずに除外した（さんま、かつお、豚肉(もも)など）。

In [ ]:
# ===== 関数セル1：データの読み込みと前処理（定義だけ。実行は下の「実行セル」で行う） =====

def load_estat_crosstab(path, encoding="cp932"):
    """e-Stat のクロス集計表 CSV を読み、縦持ち（1行=銘柄×月）の表にして返す。

    入力: path は CSV のパス。e-Stat の CSV は Shift-JIS 系なので cp932 で読む。
    戻り値: DataFrame（raw_name=銘柄名, month=YYYY-MM, value=価格[円]。欠損は NaN）
    """
    # CSV を全行リストとして読み込む（先頭には統計名などの説明行が付いている）
    with open(path, encoding=encoding) as f:
        rows = list(csv.reader(f))

    # 「/銘柄」と書かれた行が見出し行。その右側の列が銘柄名（品目名）になっている
    header_idx = next(i for i, r in enumerate(rows) if len(r) > 6 and r[6] == "/銘柄")
    names = rows[header_idx][7:]

    # 見出し行より下が月ごとのデータ行。
    # 横持ち（列=銘柄、行=月）を、1行=1銘柄×1か月 の縦持ちに組み替える
    records = []
    for r in rows[header_idx + 1:]:
        if len(r) <= 7 or not r[5]:
            continue                               # データ行でない（空行・注記）は飛ばす
        m = re.match(r"(\d{4})年(\d{1,2})月", r[5])  # 「2026年8月」から年と月を取り出す
        if not m:
            continue
        month = f"{m.group(1)}-{int(m.group(2)):02d}"   # 「2026-08」の形にそろえる
        for name, v in zip(names, r[7:]):
            # 「4,879」のカンマを除いて数値にする。「-」「...」などの欠損は NaN（空）にする
            value = pd.to_numeric(v.replace(",", ""), errors="coerce")
            records.append((name, month, value))
    return pd.DataFrame(records, columns=["raw_name", "month", "value"])


def load_extra_items(path):
    """画面から読み取った追加品目の CSV を読む。

    入力: path は estat_extra_items.csv（列: raw_name, month, value。load_estat_crosstab と同じ縦持ち）。
    戻り値: DataFrame（load_estat_crosstab の戻り値と同じ形なので、そのまま縦に連結できる）
    """
    return pd.read_csv(path, encoding="utf-8")


def clean_name(raw_name):
    """銘柄名から先頭の品目番号と【…】の注記を取り除く。

    例: "1001 うるち米(単一原料米,「コシヒカリ」)" -> "うるち米(単一原料米,「コシヒカリ」)"
    """
    name = re.sub(r"^\d+\s*", "", raw_name)   # 先頭の数字（品目番号）と空白を消す
    name = re.sub(r"【.*?】", "", name)         # 【調査終了】などの注記を消す
    return name.strip()


def build_prices(long_df, n_months, category, unit_map, use_keywords=None):
    """縦持ちデータを prices.csv の形（item_id, name, category, unit, price_yen, month）に整える。

    入力: long_df は load_estat_crosstab などの戻り値。n_months は使う月数。
          category は "食材" など。unit_map は {品目番号: 単位}。use_keywords は残す品目名の部分文字列のリスト。
    戻り値: dict {"prices": 完成した価格表, "dropped": 欠損があって除外した品目の一覧}
    """
    # 1) 直近 n_months か月だけに絞る
    months = sorted(long_df["month"].unique())[-n_months:]
    df = long_df[long_df["month"].isin(months)].copy()
    df["item_id"] = df["raw_name"].str.extract(r"^(\d+)")[0]   # 銘柄名の先頭の番号を item_id にする
    df["name"] = df["raw_name"].map(clean_name)                # 名前から番号と注記を除く

    # 2) 12か月のうち1か月でも欠損がある品目は除外する
    #    （平均などで補うと実際の価格推移とずれるので、補完はしない）
    miss = df.groupby(["item_id", "name"])["value"].apply(lambda s: s.isna().sum()).reset_index(name="欠損月数")
    dropped = miss[miss["欠損月数"] > 0][["name", "欠損月数"]].reset_index(drop=True)
    keep_ids = set(miss[miss["欠損月数"] == 0]["item_id"])
    df = df[df["item_id"].isin(keep_ids)]

    # 3) レシピで使う品目だけに絞る（銘柄名に USE_KEYWORDS のどれかが含まれるもの）
    if use_keywords:
        pattern = "|".join(map(re.escape, use_keywords))
        df = df[df["name"].str.contains(pattern)]

    # 4) 共通スキーマの列を作る（単位は UNIT_MAP から付ける。無ければ空欄）
    df["category"] = category
    df["unit"] = df["item_id"].map(unit_map).fillna("")
    df["price_yen"] = df["value"]
    out = df[["item_id", "name", "category", "unit", "price_yen", "month"]]
    out = out.sort_values(["item_id", "month"]).reset_index(drop=True)
    return {"prices": out, "dropped": dropped}

> **このセル**：食材の対応表（item_map）を作る関数と、価格推移のグラフを描く関数（ここは定義だけ）。
>
> **聞かれたら**：品目ごとに値段の水準が違うので、最初の月を100にした指数で比べている。

In [ ]:
# ===== 関数セル2：対応表の作成と、価格推移グラフ（定義だけ） =====
import matplotlib.pyplot as plt                 # グラフを描くライブラリ
from matplotlib import font_manager             # 日本語フォントを探すのに使う


def build_item_map(prices_df, short_names, unit_notes):
    """食材の対応表を作る。「レシピの食材名」と「e-Stat の銘柄名」と「単位」を1行にまとめた表。

    入力: prices_df は prices.csv の DataFrame（item_id, name, unit を使う）。
          short_names は {item_id: レシピでの食材名}。unit_notes は {item_id: (状況, 備考)}。
    戻り値: DataFrame（item_id, recipe_name, estat_name, unit, unit_status, note）
    """
    items = prices_df.drop_duplicates("item_id")[["item_id", "name", "unit"]].copy()  # 品目ごとに1行にする
    items["recipe_name"] = items["item_id"].map(short_names)                          # レシピでの呼び名
    items["unit"] = items["unit"].fillna("")
    items["unit_status"] = items["item_id"].map(lambda i: unit_notes.get(i, ("未確認", ""))[0])  # 無ければ「未確認」
    items["note"] = items["item_id"].map(lambda i: unit_notes.get(i, ("", ""))[1])
    items = items.rename(columns={"name": "estat_name"})
    return items[["item_id", "recipe_name", "estat_name", "unit", "unit_status", "note"]].reset_index(drop=True)


def make_price_index(prices_df, short_names):
    """最初の月を 100 にした価格指数を作る（品目ごとに値段の水準が違うので、比べやすくするため）。

    入力: prices_df は prices.csv の DataFrame（item_id, month, price_yen を使う）。
    戻り値: DataFrame（行=月、列=レシピでの食材名、値=指数。例: 120 なら最初の月より20%高い）
    """
    df = prices_df.copy()
    df["label"] = df["item_id"].map(short_names)
    table = df.pivot(index="month", columns="label", values="price_yen")   # 行=月、列=食材 の表にする
    return table / table.iloc[0] * 100                                     # 最初の月で割って100倍


def plot_price_index(index_df, highlight):
    """価格指数の折れ線グラフ。highlight の品目だけ色を付け、他は灰色にする。

    入力: index_df は make_price_index の戻り値。highlight は {食材名: 色}。
    戻り値: matplotlib の Figure
    """
    # グラフの日本語が文字化けしないよう、使えるフォントを上から順に探す
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Meiryo", "Yu Gothic", "MS Gothic", "Noto Sans CJK JP", "IPAexGothic", "IPAGothic"]:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    ink, ink2, grid = "#0b0b0b", "#52514e", "#e4e3de"          # 文字色・補助文字色・目盛り線の色
    fig, ax = plt.subplots(figsize=(9, 5.2), facecolor="#fcfcfb")
    ax.set_facecolor("#fcfcfb")
    x = range(len(index_df))                                     # 横軸は月の番号（0〜11）

    # 色を付けない品目は、背景として灰色の細い線で描く
    for name in index_df.columns:
        if name not in highlight:
            ax.plot(x, index_df[name], color="#c4c3bd", linewidth=1, zorder=1)
    ax.axhline(100, color=grid, linewidth=1, zorder=0)           # 基準線（指数100）

    # 強調する品目は色付きの太い線で描き、右端に「品目名 +〇%」を直接書く
    for name, color in highlight.items():
        ax.plot(x, index_df[name], color=color, linewidth=2, zorder=3, label=name)
        end = index_df[name].iloc[-1]
        ax.text(len(index_df) - 0.8, end, f"{name} {end - 100:+.0f}%", color=ink, va="center", fontsize=10)
    ax.plot([], [], color="#c4c3bd", linewidth=1, label=f"その他{len(index_df.columns) - len(highlight)}品目")  # 凡例用

    # 軸・目盛り・枠線・タイトル・出典の体裁を整える
    ticks = [0, 3, 6, 9, len(index_df) - 1]
    ax.set_xticks(ticks)
    ax.set_xticklabels([index_df.index[i] for i in ticks], color=ink2)
    ax.set_xlim(-0.3, len(index_df) + 2.4)                       # 右に余白を取り、直接ラベルの置き場にする
    ax.tick_params(axis="y", colors=ink2)
    ax.grid(axis="y", color=grid, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(grid)
    ax.set_ylabel(f"価格指数（{index_df.index[0]} = 100）", color=ink2)
    ax.set_title("食材価格の推移（東京都区部・直近12か月）", loc="left", color=ink, fontsize=13, pad=14)
    ax.legend(loc="upper left", frameon=False, fontsize=9, labelcolor=ink2)
    fig.text(0.01, 0.01, "出典：e-Stat 小売物価統計調査（動向編）を加工して作成", color=ink2, fontsize=8)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    return fig

> **このセル**：元のCSVと追加の4品目（食パン・豆腐・トマト・ねぎ）をつないで、価格表を作る。
>
> **聞かれたら**：追加の4品目は元のCSVに入っていなかったので、e-Statの画面から値を写した。結果は22品目×12か月＝264行。

In [ ]:
# ===== 実行セル1：価格表を作って中身を確認する =====
# 元のCSV（53品目ぶん）と、画面から読み取った追加4品目を、縦に連結して1つの表にする
long_df = pd.concat([load_estat_crosstab(RAW_PATH), load_extra_items(EXTRA_RAW_PATH)], ignore_index=True)
# 前処理（直近12か月・欠損品目の除外・品目の絞り込み・単位付け）を実行する
result = build_prices(long_df, N_MONTHS, CATEGORY, UNIT_MAP, USE_KEYWORDS)
# 完成した価格表と、欠損により除外した品目を確認する
prices = result["prices"]
print(prices.shape, prices["item_id"].nunique())   # （行数, 列数）と品目数。22品目×12か月=264行になるはず
print(result["dropped"])                           # 欠損があって外した品目（さんま・かつお・豚肉(もも)など）
prices.head(15)                                    # 先頭15行を表示して目で確認する

> **このセル**：価格表を `data/prices.csv` に保存する。
>
> **聞かれたら**：列はチームのルール（team_rules 第2節）どおり。

In [ ]:
# ===== 実行セル2：完成した価格表を data/prices.csv に保存する（チームの共通データ） =====
prices.to_csv(OUT_PATH, index=False, encoding="utf-8")   # index=False は行番号を書き出さない指定
print("保存しました:", OUT_PATH)

> **このセル**：食材の対応表を `data/item_map.csv` に保存する。
>
> **聞かれたら**：「レシピでの食材名 ↔ e-Statの銘柄 ↔ 単位」を1行で見られる表。

In [ ]:
# ===== 実行セル3：食材の対応表（item_map.csv）を作って保存する =====
item_map = build_item_map(prices, SHORT_NAMES, UNIT_NOTES)
OUT_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)    # 保存先フォルダが無ければ作る
item_map.to_csv(OUT_MAP_PATH, index=False, encoding="utf-8")
print("保存しました:", OUT_MAP_PATH)
item_map                                                  # 表を表示して確認する

> **このセル**：価格推移のグラフを `docs/price_trend.png` に保存して、12か月でどれだけ変わったかを出す。
>
> **聞かれたら**：グラフに出典「e-Statを加工して作成」を入れているのは、e-Statの利用規約に合わせるため。

In [ ]:
# ===== 実行セル4：価格推移グラフを作って保存し、12か月の変化率を表示する =====
price_index = make_price_index(prices, SHORT_NAMES)       # 最初の月を100にした指数の表
fig = plot_price_index(price_index, HIGHLIGHT)            # 折れ線グラフを作る
OUT_FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_FIG_PATH, dpi=150)                        # 画像として保存（資料・画面に貼る用）
print("保存しました:", OUT_FIG_PATH)
print("12か月の変化（%）:")                                # 最初の月から最後の月までに何%上がった／下がったか
print((price_index.iloc[-1] - 100).sort_values().round(1).to_string())

## 出典

政府統計の総合窓口（e-Stat）「小売物価統計調査（動向編）」（総務省統計局）を加工して作成。

- e-Stat の利用規約では、出典の記載に加えて、**編集・加工したことを別に書く**ことが求められている。国が作ったデータと誤解されないようにするため。
- 画面・資料には「出典：e-Stat 小売物価統計調査（動向編）を加工して作成」と書く。

## 確認メモ（手で埋める）

- [x] 単位は総務省の「品目・銘柄・単位」一覧で確認済み（2025年12月現在。2026年に銘柄が変わっていないかは未確認）
- [ ] 一覧のPDFで、調理パン（100g）・もやし（1kg）・納豆（1パック=150g）を目で見て確かめる
- [ ] 単位を `unit` 列のまま渡すか、1gあたりに換算して渡すかを、岩間さんに相談する
- [ ] 食パン（銘柄1021）が e-Stat の同じ表にあれば、調理パンの代わりに使う
- [ ] 外食の価格は王さんの担当。`prices.csv` に外食を足すときは `category` を「外食」にする